# ATML PA3 · Task 1 — LLM fine-tuning (Kaggle, GPU T4 ×2)

Runs every Task 1 job in dependency order on both GPUs (one job per GPU) and **resumes where the
previous session stopped**: finished jobs are skipped, training resumes from its latest recovery
checkpoint in the private HF store. Re-run (*Save & Run All*) until it prints
`REMAINING JOBS: none`.

1. First run `MODE = "smoke"` (~1 h): 20 updates of every variant + 16-example evaluations.
2. Then `MODE = "full"`. Expect 3–4 sessions in total (5 trainings ≈ 3–4 h each, two at a time;
   7 evaluations ≈ 1 h each; diagnostics, qualitative generations, summary).

Settings: *Accelerator* GPU T4 ×2, *Internet* on, *Persistence* off. Use **Save Version → Save &
Run All (Commit)** so it runs in the background (12 h limit; `HOURS` keeps a safety margin).

**Secrets** (Kaggle: *Add-ons → Secrets*; Colab: key icon): `HF_TOKEN` (write access; used for the
private checkpoint store and model downloads) and `GITHUB_TOKEN` (read access to the private repo;
not needed once the repo is public).

In [ ]:
# --- Run configuration ---
REPO = "https://github.com/rana-taqveem/atml_internal.git"
BRANCH = "main"
STORE_REPO = ""          # "" -> <your-hf-username>/atml-pa3-store (private, created automatically)
MODE = "smoke"           # "smoke" first, then "full"
HOURS = 11.3             # session budget; training jobs stop cleanly and checkpoint before it
GPUS = "0,1"
ONLY = ""                # optional comma-separated job names, e.g. "train_full,train_l2sp"

In [ ]:
# --- Setup: secrets, clone, install (identical in every task notebook) ---
import os, subprocess, sys

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient          # Kaggle
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata                       # Colab
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

os.environ["HF_TOKEN"] = secret("HF_TOKEN")
if STORE_REPO:
    os.environ["PA3_STORE_REPO"] = STORE_REPO
gh = secret("GITHUB_TOKEN")
url = REPO.replace("https://", f"https://{gh}@") if gh else REPO
# Kaggle: /kaggle/working is capped at 20 GB and saved with the notebook, so the repo and all
# checkpoints live in /tmp; only result zips are written to /kaggle/working.
KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if KAGGLE else "/content"
CLONE = "/tmp/pa3" if KAGGLE else "/content/pa3"
if not os.path.exists(CLONE):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, CLONE], check=True)
else:
    subprocess.run(["git", "-C", CLONE, "pull", "--ff-only"], check=True)
os.chdir(f"{CLONE}/assignment_03/ATML-PA3")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# --- Run all pending Task 1 jobs ---
args = ["--gpus", GPUS, "--hours", str(HOURS)]
if MODE == "smoke":
    args += ["--smoke"]
if ONLY:
    args += ["--only", ONLY]
!python -m task1_finetune.run_all {" ".join(args)}

In [ ]:
# --- Peek at results so far ---
import glob, json
for f in sorted(glob.glob("results/task1/train/*/summary.json")):
    s = json.load(open(f)); print(f.split("/")[-2], f"{s['train_hours']:.2f} h", f"peak {s.get('peak_allocated_gb', 0):.1f} GB")
for f in sorted(glob.glob("results/task1/eval/*/scores.json")):
    s = json.load(open(f)); print(f.split("/")[-2], {k: s[k] for k in ("gsm8k", "hellaswag", "boolq", "arc_easy") if k in s})
if os.path.exists("results/task1/summary/benchmarks.csv"):
    import pandas as pd; display(pd.read_csv("results/task1/summary/benchmarks.csv").round(2))

In [ ]:
ZIP_NAME, RESULTS_DIR = 'task1_results', 'results/task1'
# --- Collect: small results into a zip next to the notebook output (also mirrored in the HF store) ---
import shutil
shutil.make_archive(f"{WORK}/{ZIP_NAME}", "zip", ".", RESULTS_DIR)
if os.path.exists("logs"):
    shutil.make_archive(f"{WORK}/{ZIP_NAME}_logs", "zip", ".", "logs")
print("wrote", f"{WORK}/{ZIP_NAME}.zip")